In [1]:
#FinSentGPT model sentiment analysis after fine-tuning with specific keywords on dataset with Headline & Snippet columns

In [ ]:
import pandas as pd
from transformers import pipeline

# Define domain-specific keywords for oil, energy, and geopolitical sentiment analysis
positive_keywords = ['increase', 'growth', 'boost', 'profit', 'rise', 'gain', 'stable', 'investment']
negative_keywords = ['tensions', 'disruption', 'sanctions', 'embargo', 'decline', 'drop', 'crisis', 'loss']

# Load the FinSentGPT sentiment analysis model
sentiment_pipeline = pipeline("text-classification", model="ProsusAI/finbert")

# Load dataset
file_path = "oil_prices_headlines.csv"  # Update this path based on your file location
df = pd.read_csv(file_path, encoding="latin1")

# Check if required columns exist
if 'Headline' not in df.columns or 'Snippet' not in df.columns:
    raise ValueError("The CSV file must have 'Headline' and 'Snippet' columns for sentiment analysis!")

# Function to count keyword presence in a text
def count_keywords(text, keyword_list):
    return sum(1 for keyword in keyword_list if keyword.lower() in text.lower())

# Function to adjust sentiment labels based on keyword impact
def adjust_sentiment_label(result, text):
    label = result["label"]
    score = result["score"]

    # Count keyword occurrences
    positive_count = count_keywords(text, positive_keywords)
    negative_count = count_keywords(text, negative_keywords)

    # Adjust confidence score based on keyword presence
    if positive_count > negative_count:
        if label == "negative":
            score = max(0.01, score - 0.15)  # Reduce confidence in negative if positive keywords appear
            label = "neutral"
        elif label == "neutral":
            label = "positive" if score < 0.6 else label  # Slightly push towards positive
    elif negative_count > positive_count:
        if label == "positive":
            score = max(0.01, score - 0.15)  # Reduce confidence in positive if negative keywords appear
            label = "neutral"
        elif label == "neutral":
            label = "negative" if score < 0.6 else label  # Slightly push towards negative

    return label

# Function to analyze sentiment for both Headline and Snippet
def analyze_sentiment(text):
    result = sentiment_pipeline(text)[0]  # Get sentiment label
    adjusted_label = adjust_sentiment_label(result, text)  # Modify label based on keyword influence
    return adjusted_label  # Return only adjusted sentiment label (no confidence score)

# Apply FinSentGPT model's sentiment analysis to both headline and snippet columns
df["Headline Sentiment"] = df["Headline"].astype(str).apply(lambda x: analyze_sentiment(x))
df["Snippet Sentiment"] = df["Snippet"].astype(str).apply(lambda x: analyze_sentiment(x))

# Save the updated dataset
output_file = "oil_prices_headlines_with_keyword_sentiment.csv"
df.to_csv(output_file, index=False)

print(f"✅ Sentiment analysis completed with keyword-based adjustments! Results saved to '{output_file}'")


In [ ]:
#FinSentGPT model sentiment analysis after fine-tuning with specific keywords on dataset with Headline & Tone columns

In [ ]:
import pandas as pd
from transformers import pipeline
import numpy as np

# Define domain-specific keywords and their impact
positive_keywords = ['increase', 'growth', 'boost', 'profit', 'rise', 'gain', 'stable']
negative_keywords = ['tensions', 'disruption', 'sanctions', 'embargo', 'decline', 'drop', 'crisis', 'loss']

# Load the FinSentGPT sentiment analysis model
sentiment_pipeline = pipeline("text-classification", model="ProsusAI/finbert")

# Load dataset
df = pd.read_csv("/Users/Yashvishah/Desktop/gdelt_5_years_filtered (1).csv", encoding="latin1")

# Check if required columns exist
if 'Headline' not in df.columns:
    raise ValueError("The CSV file must have a 'Headline' column to run sentiment analysis!")
if "Tone" not in df.columns:
    raise ValueError("The CSV file must have a 'Tone' column for sentiment analysis!")

# Function to classify sentiment based on Tone
def classify_sentiment(tone):
    if tone > 2:
        return "positive"
    elif tone < -2:
        return "negative"
    else:
        return "neutral"

# Function to count keyword presence in a text
def count_keywords(text, keyword_list):
    return sum(1 for keyword in keyword_list if keyword.lower() in text.lower())

# Function to adjust sentiment scores based on keyword impact
def adjust_sentiment_score(result, text):
    label = result["label"]
    score = result["score"]

    # Count keyword occurrences
    positive_count = count_keywords(text, positive_keywords)
    negative_count = count_keywords(text, negative_keywords)

    # Adjust confidence score based on keyword presence
    if positive_count > negative_count:
        if label == "negative":
            score = max(0.01, score - 0.15)  # Reduce confidence in negative if positive keywords appear
            label = "neutral"
        elif label == "neutral":
            label = "positive" if score < 0.6 else label  # Slightly push towards positive
    elif negative_count > positive_count:
        if label == "positive":
            score = max(0.01, score - 0.15)  # Reduce confidence in positive if negative keywords appear
            label = "neutral"
        elif label == "neutral":
            label = "negative" if score < 0.6 else label  # Slightly push towards negative

    return label, score

# Apply FinSentGPT model's sentiment analysis to the headline column
def analyze_sentiment(text):
    result = sentiment_pipeline(text)[0]  # Get sentiment label and score
    adjusted_label, adjusted_score = adjust_sentiment_score(result, text)
    return adjusted_label, adjusted_score

# Apply the updated sentiment analysis function to the dataset
df[["Headline Sentiment", "Sentiment Confidence"]] = df["Headline"].astype(str).apply(lambda x: pd.Series(analyze_sentiment(x)))

# Apply the function to classify articles based on Tone column
df["Article Sentiment"] = df["Tone"].apply(classify_sentiment)

# Save the updated dataset
output_file = "gdelt_5_years_with_keyword_sentiment.csv"
df.to_csv(output_file, index=False)

print(f"✅ Sentiment analysis completed with keyword-based adjustments! Results saved to '{output_file}'")
